In [1]:
import json
import optuna
import warnings
import pandas as pd
from sklearn.metrics import root_mean_squared_error

from utils.tuning import (
    tune_hyperparameters,
    get_hyperparams_getter_function_by_model_name,
)
from utils.config import load_config
from utils.shared import get_model_by_name

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
config = load_config()

In [3]:
for model in config.models:
    for data_dim in config.sample_sizes:

        print(
            f"Start tuning {model} model for {data_dim}x{config.train_samples_num} data..."
        )

        estimator = get_model_by_name(model, config.random_seed, config.n_jobs)
        hyperparams_getter_func = get_hyperparams_getter_function_by_model_name(model)
        data = pd.read_csv(
            f"{config.data_dir}/{config.train_data_subdir}/fbm_{data_dim}x{config.train_samples_num}.csv"
        )

        study = tune_hyperparameters(
            experiment_name=config.tunning_experiment_name,
            run_name=f"{model}_{data_dim}",
            estimator=estimator,
            hyperparams_getter_func=hyperparams_getter_func,
            scoring_func=root_mean_squared_error,
            direction="minimize",
            X_train=data.drop(columns=[config.target_column]),
            y_train=data[config.target_column],
            cv_folds=config.cv_folds,
            n_trials=config.trials[model],
            cpus_to_use=config.n_jobs,
        )

        # save best hyperparameters
        with open(f"hyperparams/{model}_{data_dim}.json", "w") as f:
            json.dump(study.best_params, f, indent=4)

        print(f"Tunning is finished.")

Start tuning XGBoost model for 25x100000 data...


  0%|          | 0/80 [00:00<?, ?it/s]

Tunning is finished.
Start tuning XGBoost model for 50x100000 data...


  0%|          | 0/80 [00:00<?, ?it/s]

Tunning is finished.
Start tuning XGBoost model for 100x100000 data...


  0%|          | 0/80 [00:00<?, ?it/s]

Tunning is finished.
